# 00 — Is this kernel actually what I think it is?

Six checks, in order, each one cheap. If a cell fails, stop there: everything
below it depends on it.

This is also a rough draft of a tutorial to my work, and it serves as documentation,
and a source code to what I'm doing in my research.

1. the kernel's environment,
2. TensorFlow can see the GPU,
3. `bnhgq2` imports, and from the copy `lab.sh sync` pushed,
4. one `.h5` file loads with the round-14 `(N, 3)` inputs,
5. the model the config describes builds,
6. it runs a forward pass.

### 1 — environment

In [ ]:
import os, sys, shutil, platform, subprocess

print("python :", sys.version.split()[0])
print("prefix :", sys.prefix)
print("host   :", platform.platform(), "|", platform.machine())
for k in ("KERAS_BACKEND", "BNHGQ2_TRAIN_DATA", "BNHGQ2_OUT_ROOT",
          "BNHGQ2_STORE", "PYTHONPATH", "WANDB_MODE", "WANDB_PROJECT"):
    print(f"  {k:18s} = {os.environ.get(k)}")

# nvidia-smi exists in the pod and not on the laptop; absence is information,
# not an error - section 2 is what actually decides which device we get.
if shutil.which("nvidia-smi"):
    print(subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
        capture_output=True, text=True).stdout)
else:
    print("\nno nvidia-smi on this host (expected on the laptop)")

### 2 — which device are we actually on?

On the NRP pod a GPU is the whole point, so its absence is a hard failure — a kernel that quietly fell back to CPU would be a pod worth deleting. On the laptop CPU is the expected answer: at 18,657 parameters the measured smoke train is *faster* on the M5's CPU than on the pod's 2080 Ti, because dispatch overhead dominates long before the FLOPs do.

In [ ]:
import tensorflow as tf

print("tensorflow", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus or "none — running on CPU")
if platform.system() == "Linux":
    assert gpus, "no GPU visible to TensorFlow (on the pod this is fatal)"

### 3 — our code imports, and from the synced copy

In [ ]:
import inspect
import bnhgq2
from bnhgq2 import config as bncfg, data as bndata, qat
from bnhgq2 import train as bntrain

print("bnhgq2 :", os.path.dirname(inspect.getfile(bnhgq2)))
print("PROJECT_ROOT :", bncfg.PROJECT_ROOT)

### 4 — a small sample: one `.h5` file, round-14 `(N, 3)` inputs

`load_train_data` is the same function the cluster training job calls. `max_files=1` is the only thing making this small.

In [ ]:
import numpy as np

CONFIG = os.path.join(os.environ["PYTHONPATH"], "configs", "r14-l1x3-n8-w1a8.json")
cfg = bncfg.load_config(CONFIG)
A = cfg["arch"]
print(cfg["name"], "| n_part", A["n_part"], "| features", A["features"],
      "| weight", cfg["quant"]["weight"], "| act_bits", cfg["quant"]["act_bits"])

X, Y, nfiles = bntrain.load_train_data(
    os.environ["BNHGQ2_TRAIN_DATA"], A["n_part"],
    max_files=1, features=A["features"])

print("files :", nfiles)
print("X     :", X.shape, X.dtype)
print("Y     :", Y.shape, Y.dtype)
print("class counts:", dict(zip(bndata.CLASS_NICE, Y.sum(0).astype(int))))

### 5 — build the model the config describes

In [ ]:
model, taps = qat.build_qat_model(cfg, seed=1)
model.summary()
# count_params() is the TOTAL (trainable + non-trainable); the summary above
# splits them. For r14-l1x3-n8-w1a8 the total is 18,657.
print("total parameters:", model.count_params())

### 6 — one forward pass

In [ ]:
import time

xb = X[:256]
_ = model(xb[:8], training=False)                     # trace/compile once
t0 = time.time()
logits = model(xb, training=False)
dt = time.time() - t0
print("logits :", np.asarray(logits).shape)
print(f"forward pass on 256 jets: {dt * 1e3:.1f} ms")

---
If all six passed, the kernel is real, the data is reachable and the model
builds. Next: **`01_smoke_train.ipynb`** actually trains it (3 epochs, 2 files).